In [1]:
# Setup
import json

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, grid_search_forecaster

MODEL_LABEL = "ARX + Ridge"
MAX_CONTEXT = 8192
MAX_HORIZON = 1024
print(f"Using {MODEL_LABEL}")

Using ARX + Ridge


In [2]:
# Load data
import json

DATA_FILE = "tep_sp_spaced_6min_constdist_noisy.h5"


def read_names(ds):
    return [n.decode() if isinstance(n, bytes) else str(n) for n in ds.attrs["names"]]


groups = {}
with h5py.File(DATA_FILE, "r") as f:
    time_h = f["t"][()] / 3600
    for key in ["xmeas", "xmv", "dist", "sp"]:
        groups[key] = (f[key][()], read_names(f[key]))
    if "y" in f:   # true valve positions
        groups["valve"] = (f["y"][:, 38:50], read_names(f["y"])[38:50])
    config = json.loads(f.attrs["config"])
SAMPLE_MIN = config["record_every"] / 60   # sampling interval (min)

nominal = {int(k): v for k, v in config["setpoints"].items()}
CHANGED = sorted({k for k, _, _ in config["setpoint_schedule"]})   # set points changed in this run
t_test = config.get("t_test")                                    # start of the test changes (h)

sp = groups["sp"][0]
off = {k: np.abs(sp[:, k - 1] - nominal[k]) > 1e-9 for k in CHANGED}
sp_change = np.zeros(len(time_h), dtype=int)
for k in reversed(CHANGED):
    sp_change[off[k]] = k

data = {"time_h": time_h, "sp_change": sp_change}
COLUMNS, LABELS = {}, {}
for group, (values, labels) in groups.items():
    COLUMNS[group] = [f"{group}_{i}" for i in range(1, values.shape[1] + 1)]
    for col, v, label in zip(COLUMNS[group], values.T, labels):
        data[col] = v
        LABELS[col] = label

df = pd.DataFrame(data)
df.insert(0, "item_id", "tep")
df.insert(1, "timestamp", pd.date_range("2026-01-01", periods=len(df), freq=f"{config['record_every']}s"))

TARGET_COLUMNS = COLUMNS["xmeas"] + COLUMNS["xmv"]
print(f"{len(df)} samples ({time_h[-1]:.0f} h), {len(TARGET_COLUMNS)} targets, covariate groups: "
      + ", ".join(f"{g} ({len(COLUMNS[g])})" for g in COLUMNS if g not in ["xmeas", "xmv"]))
print(f"changed set points: {CHANGED}, test part from {t_test} h")
df.head()

6798 samples (680 h), 53 targets, covariate groups: dist (12), sp (20), valve (12)
changed set points: [5, 7, 8, 11, 13, 14, 15, 17, 18, 19, 20], test part from 580.6 h


,item_id,timestamp,time_h,sp_change,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,...,valve_3,valve_4,valve_5,valve_6,valve_7,valve_8,valve_9,valve_10,valve_11,valve_12
0,tep,2026-01-01 00:00:00,0.1,0,0.250570,3646.911201,4529.162679,9.362211,26.955963,42.373414,...,24.683121,61.479204,22.363783,40.094634,38.425616,46.451781,47.454458,41.014715,17.044840,50.0
1,tep,2026-01-01 00:06:00,0.2,0,0.251641,3685.782817,4532.305731,9.392836,27.191772,42.304758,...,24.669825,61.225171,22.292722,39.989795,38.837808,46.079812,47.453757,40.959467,19.485093,50.0
2,tep,2026-01-01 00:12:00,0.3,0,0.224988,3649.653552,4485.108841,9.373259,26.671660,42.429623,...,21.911699,61.458982,22.123348,40.747079,37.087973,47.818568,47.557016,40.224623,16.873591,50.0
3,tep,2026-01-01 00:18:00,0.4,0,0.218616,3639.422517,4472.533021,9.491735,26.897842,42.106417,...,21.639985,61.548652,22.355574,40.738591,34.571177,46.714696,47.677321,40.833625,17.571302,50.0
4,tep,2026-01-01 00:24:00,0.5,0,0.259312,3666.070767,4583.076516,9.291715,27.026806,42.234145,...,25.624424,61.177210,21.877342,42.039533,38.116623,48.398896,47.830919,40.202709,18.061803,50.0


In [3]:
# Test events and forecast windows
events = []   # (start index, end index, set point, new value)
for k in CHANGED:
    on = off[k]
    for i in range(len(on)):
        if on[i] and (i == 0 or not on[i - 1]):
            start = i
        if on[i] and (i == len(on) - 1 or not on[i + 1]):
            events.append((start, i, k, sp[start, k - 1]))
events.sort()

BUFFER = 20   # nominal samples before the event at the start of each forecast
# Samples where a disturbance starts to ramp to a new operating point, none for most datasets
ramp_starts = sorted({int(np.searchsorted(time_h, t0)) for _, t0, _, _ in config.get("dist_schedule") or []})

# One forecast per test event, from BUFFER samples before the event up to the start of the next event or of
# the next disturbance ramp
rows = []
for n, (s, e, k, v) in enumerate(events):
    if t_test is None or time_h[s] < t_test:
        continue
    split = max(s - BUFFER, 0)
    end = events[n + 1][0] if n + 1 < len(events) else len(df)
    end = min([end] + [r for r in ramp_starts if r > e])
    if end <= e:
        raise ValueError(f"event {n} is still running when the next event starts")
    horizon = min(end - split, MAX_HORIZON)
    rows.append({
        "event": n,
        "set point": f"SP {k} {LABELS[f'sp_{k}']}",
        "change (%)": 100 * (v - nominal[k]) / nominal[k],
        "start (h)": time_h[split],
        "horizon (h)": horizon * SAMPLE_MIN / 60,
        "split": split,
        "horizon": horizon,
    })
windows = pd.DataFrame(rows).set_index("event")

EXCLUDED = []   # test events left out of the scores
windows = windows.drop(EXCLUDED)
print(f"{len(windows)} test events")
windows.round(2)

11 test events


,set point,change (%),start (h),horizon (h),split,horizon
event,,,,,,
66,SP 11 Stripper underflow (m3/h),1.74,578.6,10.9,5785,109
67,SP 5 Recycle flow (kscmh),-1.86,587.5,8.9,5874,89
68,SP 14 Reactor feed D (mol%),2.91,594.4,12.3,5943,123
69,SP 20 Product E (mol%),-7.18,604.7,12.8,6046,128
70,SP 18 Reactor temperature (C),0.62,615.5,6.5,6154,65
71,SP 15 Reactor feed E (mol%),-3.99,620.0,12.1,6199,121
72,SP 13 Reactor feed A (mol%),-1.86,630.1,12.8,6300,128
73,SP 17 Reactor level (%),-1.67,640.9,10.0,6408,100
74,SP 19 Purge B (mol%),7.23,648.9,12.8,6488,128


In [4]:
# Covariates and scored variables
USE_DIST = True                  # disturbance signals (12)
USE_SP = True                    # changed set points (11)
USE_VALVE = False                # true valve positions (12)
DROPPED = []
PAST_ONLY = []
COVARIATES_KNOWN_FUTURE = True

ALL_COVARIATES = []
for group, use in [("dist", USE_DIST), ("sp", USE_SP), ("valve", USE_VALVE)]:
    if use and group in COLUMNS:
        ALL_COVARIATES += [f"sp_{k}" for k in CHANGED] if group == "sp" else COLUMNS[group]
ALL_COVARIATES = [c for c in ALL_COVARIATES if c not in DROPPED]


def covariates(window_df):
    """Covariates for one forecast window, leaving out those that are constant over it."""
    used = [c for c in ALL_COVARIATES if window_df[c].nunique() > 1]
    future = [c for c in used if COVARIATES_KNOWN_FUTURE and c not in PAST_ONLY]
    past_only = [c for c in used if c not in future]
    return used, future, past_only


EXCLUDED_VARIABLES = ["xmeas_37"]   # Product D, about 0.017 mol%: near zero, so its percentage error explodes
SCORED = [c for c in COLUMNS["xmeas"] if c not in EXCLUDED_VARIABLES]   # measurements only, no valves
SCALE_FILE = "tep_sp_spaced_6min_noisy.h5"   # dataset whose standard deviation scales the errors, None for this dataset
if SCALE_FILE is None:
    SCALE = df[SCORED].std()    # standard deviation over the whole dataset
else:   # same scale for several datasets, so their scaled scores are comparable
    with h5py.File(SCALE_FILE, "r") as f:
        SCALE = pd.DataFrame(f["xmeas"][()], columns=COLUMNS["xmeas"])[SCORED].std()
    print(f"errors scaled by the standard deviation of {SCALE_FILE}")
print(f"scored variables ({len(SCORED)}): {', '.join(SCORED)}")
print(f"covariates ({len(ALL_COVARIATES)}): {', '.join(ALL_COVARIATES) or '-'}")

errors scaled by the standard deviation of tep_sp_spaced_6min_noisy.h5
scored variables (40): xmeas_1, xmeas_2, xmeas_3, xmeas_4, xmeas_5, xmeas_6, xmeas_7, xmeas_8, xmeas_9, xmeas_10, xmeas_11, xmeas_12, xmeas_13, xmeas_14, xmeas_15, xmeas_16, xmeas_17, xmeas_18, xmeas_19, xmeas_20, xmeas_21, xmeas_22, xmeas_23, xmeas_24, xmeas_25, xmeas_26, xmeas_27, xmeas_28, xmeas_29, xmeas_30, xmeas_31, xmeas_32, xmeas_33, xmeas_34, xmeas_35, xmeas_36, xmeas_38, xmeas_39, xmeas_40, xmeas_41
covariates (23): dist_1, dist_2, dist_3, dist_4, dist_5, dist_6, dist_7, dist_8, dist_9, dist_10, dist_11, dist_12, sp_5, sp_7, sp_8, sp_11, sp_13, sp_14, sp_15, sp_17, sp_18, sp_19, sp_20


In [5]:
# Metrics
QUANTILES = [0.1, 0.5, 0.9]
ALPHA = round(1 - (QUANTILES[-1] - QUANTILES[0]), 6)   # 0.2 for the 10-90% interval
METRICS = ["scaled MAE", "MAPE (%)", "coverage", "scaled Winkler"]


def winkler(actual, lower, upper, alpha):
    """Winkler interval score at every step."""
    width = upper - lower
    below = width + 2 / alpha * (lower - actual)
    above = width + 2 / alpha * (actual - upper)
    return np.where(actual < lower, below, np.where(actual > upper, above, width))


def score_event(forecast, truth_df):
    """Metrics per scored variable for one forecast, forecast[col] = (point forecast, lower, upper)."""
    rows = {}
    for col in SCORED:
        pred, lower, upper = forecast[col]
        actual = truth_df[col].to_numpy()
        rows[col] = {
            "scaled MAE": np.mean(np.abs(pred - actual)) / SCALE[col],
            "MAPE (%)": 100 * np.mean(np.abs(actual - pred) / np.abs(actual)),
            "coverage": np.mean((actual >= lower) & (actual <= upper)),
            "scaled Winkler": np.mean(winkler(actual, lower, upper, ALPHA)) / SCALE[col],
        }
    return pd.DataFrame(rows).T

In [6]:
# ARX inputs and training, once for all events
SP_LAGS = 10
LAGS_GRID = [5, 10, 20, 30, 50]               # target lags tried
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]    # Ridge strengths tried
VALIDATION = 0.3
CV_STEPS = int(windows["horizon"].max())      # backtest with the longest forecast

# Covariate columns: the covariates known over the forecast, with their lags
EXOG_COVARIATES = [c for c in ALL_COVARIATES if COVARIATES_KNOWN_FUTURE and c not in PAST_ONLY]
exog_columns = {}
for col in EXOG_COVARIATES:
    exog_columns[col] = df[col]
    for lag in range(1, SP_LAGS + 1):
        exog_columns[f"{col}_lag_{lag}"] = df[col].shift(lag)
exog_df = pd.DataFrame(exog_columns, index=df.index)
exog_df = exog_df.bfill()
EXOG_COLUMNS = list(exog_df.columns)

# Training data, everything before the first forecast starts
TRAIN_END = int(windows["split"].min())
train_df = df.iloc[:TRAIN_END]
train_exog = exog_df.iloc[:TRAIN_END] if EXOG_COLUMNS else None
print(f"Training on samples 0-{TRAIN_END - 1} (0.0-{time_h[TRAIN_END - 1]:.1f} h), "
      f"{len(EXOG_COLUMNS)} covariate columns ({len(EXOG_COVARIATES)} covariates x {SP_LAGS + 1} values)")

# Choose the target lags and Ridge strength per scored variable, then fit
cv = TimeSeriesFold(steps=CV_STEPS, initial_train_size=int(TRAIN_END * (1 - VALIDATION)), refit=False, verbose=False)
forecasters = {}
rows = []
for col in SCORED:
    results = grid_search_forecaster(
        ForecasterRecursive(Ridge(), lags=LAGS_GRID[0], transformer_y=StandardScaler(),
                            transformer_exog=StandardScaler()),
        y=train_df[col],
        exog=train_exog,
        cv=cv,
        param_grid={"alpha": ALPHA_GRID},
        lags_grid=LAGS_GRID,
        metric="mean_absolute_error",
        return_best=False,
        show_progress=False,
        suppress_warnings=True,
    )
    lags, alpha = len(results.iloc[0]["lags"]), results.iloc[0]["alpha"]
    forecaster = ForecasterRecursive(Ridge(alpha=alpha), lags=lags, transformer_y=StandardScaler(),
                                     transformer_exog=StandardScaler())
    forecaster.fit(y=train_df[col], exog=train_exog, store_in_sample_residuals=True, suppress_warnings=True)
    forecasters[col] = forecaster
    rows.append({"variable": col, "description": LABELS[col], "lags": lags, "alpha": alpha})

chosen = pd.DataFrame(rows).set_index("variable")
print("Number of targets per chosen lags:")
print(chosen["lags"].value_counts().sort_index().to_string())
chosen

Training on samples 0-5784 (0.0-578.5 h), 253 covariate columns (23 covariates x 11 values)


Number of targets per chosen lags:
lags
5      4
10     7
20     7
30    11
50    11


,description,lags,alpha
variable,,,
xmeas_1,A feed (kscmh),50,0.01
xmeas_2,D feed (kg/h),50,100.00
xmeas_3,E feed (kg/h),30,0.01
xmeas_4,A and C feed (kscmh),30,0.01
xmeas_5,Recycle flow (kscmh),10,100.00
xmeas_6,Reactor feed rate (kscmh),50,10.00
xmeas_7,Reactor pressure (kPa),20,0.01
xmeas_8,Reactor level (%),30,10.00
xmeas_9,Reactor temperature (C),10,10.00


In [7]:
# ARX + Ridge forecast of one window
def run_forecast(context_df, truth_df, used, future, past_only):
    start = truth_df.index[0]
    future_exog = exog_df.iloc[start:start + len(truth_df)] if EXOG_COLUMNS else None   # keeps the RangeIndex
    forecast = {}
    for col in SCORED:
        last_window = context_df[col]
        prediction = forecasters[col].predict(steps=len(truth_df), last_window=last_window, exog=future_exog,
                                              suppress_warnings=True)
        interval = forecasters[col].predict_quantiles(steps=len(truth_df), last_window=last_window,
                                                      exog=future_exog, quantiles=QUANTILES, suppress_warnings=True)
        forecast[col] = (prediction.to_numpy(), interval[f"q_{QUANTILES[0]}"].to_numpy(),
                         interval[f"q_{QUANTILES[-1]}"].to_numpy())
    return forecast

In [8]:
# Forecast every test event
per_variable = {}   # event -> metrics per scored variable
for n, w in windows.iterrows():
    split, horizon = int(w["split"]), int(w["horizon"])
    context_used = min(split, MAX_CONTEXT)
    context_df = df.iloc[split - context_used:split]
    truth_df = df.iloc[split:split + horizon]
    used, future, past_only = covariates(pd.concat([context_df, truth_df]))

    forecast = run_forecast(context_df, truth_df, used, future, past_only)
    per_variable[n] = score_event(forecast, truth_df)
    means = per_variable[n].mean()
    print(f"event {n:2d} {w['set point']}: " + ", ".join(f"{m} {means[m]:.3f}" for m in METRICS))

event 66 SP 11 Stripper underflow (m3/h): scaled MAE 0.536, MAPE (%) 2.176, coverage 0.777, scaled Winkler 2.270


event 67 SP 5 Recycle flow (kscmh): scaled MAE 0.497, MAPE (%) 2.111, coverage 0.811, scaled Winkler 2.198


event 68 SP 14 Reactor feed D (mol%): scaled MAE 0.426, MAPE (%) 1.936, coverage 0.862, scaled Winkler 2.084


event 69 SP 20 Product E (mol%): scaled MAE 0.413, MAPE (%) 1.722, coverage 0.894, scaled Winkler 1.993


event 70 SP 18 Reactor temperature (C): scaled MAE 0.502, MAPE (%) 2.194, coverage 0.806, scaled Winkler 2.228


event 71 SP 15 Reactor feed E (mol%): scaled MAE 0.534, MAPE (%) 2.619, coverage 0.783, scaled Winkler 2.339


event 72 SP 13 Reactor feed A (mol%): scaled MAE 0.419, MAPE (%) 1.865, coverage 0.884, scaled Winkler 1.985


event 73 SP 17 Reactor level (%): scaled MAE 0.481, MAPE (%) 2.290, coverage 0.839, scaled Winkler 2.116


event 74 SP 19 Purge B (mol%): scaled MAE 0.494, MAPE (%) 2.229, coverage 0.824, scaled Winkler 2.206


event 75 SP 7 Separator level (%): scaled MAE 0.481, MAPE (%) 2.002, coverage 0.838, scaled Winkler 2.142


event 76 SP 8 Stripper level (%): scaled MAE 0.543, MAPE (%) 2.454, coverage 0.766, scaled Winkler 2.348


In [9]:
# Scores per event, averaged over the scored variables
event_scores = pd.DataFrame({n: s.mean() for n, s in per_variable.items()}).T[METRICS]
event_table = windows[["set point", "change (%)", "start (h)", "horizon (h)"]].join(event_scores)
print(f"{MODEL_LABEL}: mean over {len(SCORED)} measurements per event")
event_table.round(3)

ARX + Ridge: mean over 40 measurements per event


,set point,change (%),start (h),horizon (h),scaled MAE,MAPE (%),coverage,scaled Winkler
event,,,,,,,,
66,SP 11 Stripper underflow (m3/h),1.743,578.6,10.9,0.536,2.176,0.777,2.270
67,SP 5 Recycle flow (kscmh),-1.859,587.5,8.9,0.497,2.111,0.811,2.198
68,SP 14 Reactor feed D (mol%),2.906,594.4,12.3,0.426,1.936,0.862,2.084
69,SP 20 Product E (mol%),-7.180,604.7,12.8,0.413,1.722,0.894,1.993
70,SP 18 Reactor temperature (C),0.623,615.5,6.5,0.502,2.194,0.806,2.228
71,SP 15 Reactor feed E (mol%),-3.994,620.0,12.1,0.534,2.619,0.783,2.339
72,SP 13 Reactor feed A (mol%),-1.864,630.1,12.8,0.419,1.865,0.884,1.985
73,SP 17 Reactor level (%),-1.667,640.9,10.0,0.481,2.290,0.839,2.116
74,SP 19 Purge B (mol%),7.234,648.9,12.8,0.494,2.229,0.824,2.206


In [10]:
# Average over all test events
final = event_scores.agg(["mean", "std"])   # mean and standard deviation over the test events
print(f"{MODEL_LABEL}: mean over {len(event_scores)} test events")
final.round(3)

ARX + Ridge: mean over 11 test events


,scaled MAE,MAPE (%),coverage,scaled Winkler
mean,0.484,2.145,0.826,2.174
std,0.047,0.260,0.042,0.123


In [11]:
# Save scores
import os

RESULTS_FILE = "Results/Loop_setpointforecast_arx_constdist_futuredist.h5"

os.makedirs("Results", exist_ok=True)
with h5py.File(RESULTS_FILE, "w") as f:
    f.attrs["model"] = MODEL_LABEL
    f.attrs["settings"] = json.dumps({
        "data_file": DATA_FILE,
        "buffer": BUFFER,
        "quantiles": QUANTILES,
        "scored": SCORED,
        "covariates": ALL_COVARIATES,
        "past_only": PAST_ONLY,
        "excluded_events": EXCLUDED,
        "excluded_variables": EXCLUDED_VARIABLES,
        "scale_file": SCALE_FILE or DATA_FILE,
    })
    f.create_dataset("event", data=event_scores.index.to_numpy())
    f.create_dataset("split", data=windows.loc[event_scores.index, "split"].to_numpy())
    f.create_dataset("horizon", data=windows.loc[event_scores.index, "horizon"].to_numpy())
    for name in METRICS:
        f.create_dataset(f"per_event/{name}", data=event_scores[name].to_numpy())
        f.create_dataset(f"average/{name}", data=final.loc["mean", name])
        f.create_dataset(f"std/{name}", data=final.loc["std", name])   # over the test events
        f.create_dataset(f"per_variable/{name}",   # events x scored variables
                         data=np.array([per_variable[n].loc[SCORED, name].to_numpy() for n in event_scores.index]))
print(f"Saved {MODEL_LABEL} scores to {RESULTS_FILE}")

Saved ARX + Ridge scores to Results/Loop_setpointforecast_arx_constdist_futuredist.h5
